# DECIMA quick start

Ages, metallicities, distances and extinctions for five Xiang+24 stars. Set `DECIMA_DATA_DIR` if the models are not downloaded automatically yet.

In [ ]:
import matplotlib.pyplot as plt
import decima

# the table uses the catalog's own column names: map the canonical ones to them
columns = {
    "id": "GAIAEDR3_SOURCE_ID_1", "ra": "RAJ2000", "dec": "DEJ2000",
    "G": "GAIA_G", "BP": "GAIA_BP", "RP": "GAIA_RP", "G_error": "E_GAIA_G", "BP_error": "E_GAIA_BP", "RP_error": "E_GAIA_RP",
    "J": "TWOMASS_J", "H": "TWOMASS_H", "Ks": "TWOMASS_K", "J_error": "j_msigcom", "H_error": "h_msigcom", "Ks_error": "ks_msigcom",
    "parallax": "PARALLAX", "parallax_error": "E_PARALLAX", "teff": "TEFF", "teff_error": "E_TEFF",
    "logg": "LOGG", "logg_error": "E_LOGG", "feh": "FEH", "feh_error": "E_FEH", "alpha_fe": "ALPHA_FE", "alpha_fe_error": "E_ALPHA_FE",
}

In [ ]:
cfg = decima.Config(n_latent_x=100, n_theta=60, n_keep=1000, extinction="F19", ak_inflate=True)
summary, samples, rejected = decima.infer("../tests/data/reference_stars.ecsv", columns, cfg)
summary["id", "age_p16", "age_p50", "age_p84", "feh_p50", "distance_p50", "A0_p50", "ess_frac"]

A custom extinction law: constant R_b, here the Schlafly & Finkbeiner (2018) values for the bands used.

In [ ]:
law = decima.FixedLaw({"G": 2.294, "BP": 3.046, "RP": 1.737, "J": 0.632, "H": 0.318, "Ks": 0.158})
summary_ss, samples_ss, _ = decima.infer("../tests/data/reference_stars.ecsv", columns,
                                         decima.Config(n_keep=1000, extinction=law))

names = decima.quantity_names(cfg)
fig, axes = plt.subplots(1, len(summary), figsize=(3 * len(summary), 2.5), sharey=True)
for i, ax in enumerate(axes):
    ax.hist(samples[i, :, names.index("age")], bins=30, histtype="step", label="F19")
    ax.hist(samples_ss[i, :, names.index("age")], bins=30, histtype="step", label="fixed R_b")
    ax.set_xlabel("age [Gyr]")
axes[0].legend()
plt.show()